# Cryptarithmetic Puzzle Solver with Constraint Satisfaction

A CSP formulation for cryptarithmetic puzzles using constraints for non-zero leading digits, unique assignments, and arithmetic consistency.

**Topics:** Constraint Satisfaction Problems, Backtracking search, Constraint design, Cryptarithmetic puzzles

> Clean portfolio version: official problem statements, grading instructions, and personal/student identifiers were removed. The remaining code represents my implementation and learning outcomes from an undergraduate Artificial Intelligence course.


In [ ]:
# Set your student number
student_number = '97102271'
name = 'Shayan'
last_name = 'Ghalehdar'

### Example


## Note


## Input
### Sample Input


## Output
### Sample Output


## Your code


### Define Constraint (20 points)


In [ ]:
import sys
sys.setrecursionlimit(5000000)

In [ ]:
import numpy as np


class Constraint:
    """
    Base class for all constraints.
    """

    def __init__(self, ops: list[str] , variables: list[str], base : int) -> None:
        """
        Initializes the constraint.

        Args:
            variable: name of the variable
        """
        
        self.ops = ops
        self.variables = variables
        self.base = base        
        
    def satisfied(self, assignment: dict[str, int]) -> bool:
        """
        Checks if the constraint is satisfied.

        Args:
            assignment: dictionary of the assignment of the variables to the values

        Returns:
            False if the constraint is not satisfied, True otherwise (including if the variable is not assigned)
        """
        return True


class NotZeroConstraint(Constraint):
    """"
    Constraint that checks if the variable is not zero.
    """

    def satisfied(self, assignment: dict[str, int]) -> bool:
        for i in range(len(self.ops)):
            if assignment.get(str(self.ops[i][0]))==0:
                return False
        return True


class UniqueConstraint(Constraint):
    """
    Constraint that checks if all the variables has unique values.
    """

    def __init__(self) -> None:
        pass

    def satisfied(self, assignment: dict[str, int]) -> bool:
        val= list(assignment.values())
        while None in val:
            val.remove(None)
        return len(val) == len(set(val))

class SumConstraint(Constraint):
    """
    Constraint that checks if the sum of the operands is equal to the result.
    """

    def __init__(self) -> None:
        pass

    def satisfied(self, assignment: dict[str, int]) -> bool:
        #TODO: implement this method
        res = self.ops[-1]
        carry=0
        for i in range(1,len(res)+1): #i:1...5            
            opsum=0
            
            nonefound=0
            for j in range(len(self.ops)-1):
                if(i<=len(self.ops[j])):
                    if assignment.get(self.ops[j][len(self.ops[j])-i]) == None:                        
                        nonefound=1
                        break
                    opsum = opsum + assignment.get(self.ops[j][len(self.ops[j])-i])
            
            if nonefound or assignment.get(self.ops[len(self.ops)-1][len(self.ops[len(self.ops)-1])-i])==None:
                break
            if opsum+carry>=self.base:
                if (opsum+carry)%self.base!=assignment.get(res[len(res)-i]):
                    return False
                carry=(opsum+carry-((opsum+carry)%self.base))/self.base
            else:
                if opsum+carry!=assignment.get(res[len(res)-i]):
                    return False
                carry=0
        return True

### Define a CSP class (50 points)


In [ ]:
class CSP:
    """
    CSP class that contains the variables, constraints and base of the problem.
    """

    def __init__(self, ops: list[str], variables: list[str], dom: list[list], reset_dom: list[list], base: int , assignment: dict[str, int]):
        
        self.ops = ops
        self.base = base
        self.variables = variables
        self.dom = dom  
        self.assignment = assignment
        self.reset_dom = reset_dom
        

    def consistent(self, assignment: dict[str, int]) -> bool:
        """
        Checks if the assignment is consistent with the constraints.

        Args:
            assignment: dictionary of the assignment of the variables to the values.

        Returns:
            False if the assignment is not consistent, True otherwise
        """
        
        dht = Constraint(self.ops, self.variables, self.base)
        if NotZeroConstraint.satisfied(dht,assignment) and UniqueConstraint.satisfied(dht,assignment) and SumConstraint.satisfied(dht,assignment):
            return True
        else:
            return False

    def backtracking_search(self, assignment):
        """
        Backtracking search algorithm.

        Args:
            assignment: dictionary of the assignment of the variables to the values. assignment is modified in place.

        Returns:
            True if a solution is found, False otherwise
        """
        #TODO: implement this method
        
        if None in set(list(assignment.values())):
            last_assgined_index = list(assignment.values()).index(None)-1
        else:
            last_assgined_index = len(assignment)-1
        
        last_assgined_variable = list(assignment)[last_assgined_index]
        unused_dom =[x for x in self.dom[last_assgined_index][:-1] if x not in set(list(assignment.values()))]
        if unused_dom != []:
            assignment[last_assgined_variable] = list(unused_dom)[0]
            filler(self.ops,assignment,self.base)
            self.dom[last_assgined_index].remove(assignment[last_assgined_variable])
            backtracking_search(self)
        else:
            assignment[last_assgined_variable] = None
            import copy
            self.dom[last_assgined_index] = copy.deepcopy(self.reset_dom[last_assgined_index])
            CSP.backtracking_search(self,assignment)
            
    
        
                
                        
        

### Build CSP and solve it! (20 points)


In [ ]:
def get_problem(operands: list[str], result: str, base: int = 10) -> CSP:
    """
    Creates a CSP problem from the operands and the result. The variables are the unique characters in the operands and the result.
    The constraints should be defined usinng NotZeroConstraint for leftmost digits, a UniqueConstraint for all variables and a SumConstraint for the problem
    multiple partial SumConstraint can be used for bigger search spaces and base > 10.

    Args:
        operands: list of the operands
        result: result of the sum
        base: base of the numbers

    Returns:
        CSP problem
    """
    domlist=[]
    for i in range(base):
        domlist.append(i)
    
    ops = operands + [result]
    variables=[]
    for i in range(len(result)):
        for j in range(len(ops)):
            if i<len(ops[j]):
                if ops[j][-i-1] not in ["\n"," "] and ops[j][-i-1] not in variables:
                    variables.append(ops[j][-i-1])      
    
    ass={}
    for i in range(len(variables)):
        ass[variables[i]]= None
        
    dom = [domlist] * len(variables)
    for i in range(len(variables)):
        dom[i]=dom[i]+[variables[i]]
    
    notzero = []
    for i in range(len(ops)):
        if ops[i][0] not in notzero:
            notzero.append(ops[i][0])
    for i in range(len(variables)):
        if dom[i][-1] in notzero:
            dom[i].remove(0)
    
    import copy
    reset_dom = copy.deepcopy(dom)
    return CSP(ops, variables, dom, reset_dom, base, ass)


In [ ]:
def filler(ops, assignment, base):
    carry=0
    for i in range(len(ops[-1])):
        s=carry
        br=0
        carry=0
        for j in range(len(ops)-1):
            if i<len(ops[j]):
                if assignment[ops[j][-i-1]]==None:
                    br = 1
                else:
                    s=s+assignment[ops[j][-i-1]]

        if br:
            break
        while s>=base:
            s=s-base
            carry=carry+1
        if assignment[ops[len(ops)-1][-i-1]]==None:
            assignment[ops[len(ops)-1][-i-1]] = s


In [ ]:
def backtracking_search(csp: CSP) -> dict[str, int]:
    """
    Backtracking search algorithm.

    Args:
        csp: CSP problem
    
    Returns:
        dictionary of the assignment of the variables to the values if a solution is found, None otherwise
    """
    
    if None in set(list(csp.assignment.values())):
        first_unassgined_index = list(csp.assignment.values()).index(None)
    else:
        if CSP.consistent(csp,csp.assignment):
            return csp.assignment

    if CSP.consistent(csp,csp.assignment) and (None in set(list(csp.assignment.values()))):
        first_unassgined_variable = list(csp.assignment)[first_unassgined_index]
        unused_dom = [x for x in csp.dom[first_unassgined_index][:-1] if x not in set(list(csp.assignment.values()))]
        csp.assignment[first_unassgined_variable] = list(unused_dom)[0]
        filler(csp.ops,csp.assignment,csp.base)
        csp.dom[first_unassgined_index].remove(csp.assignment[first_unassgined_variable])
        backtracking_search(csp)        
    else:
        CSP.backtracking_search(csp,csp.assignment)
    
    return csp.assignment

In [ ]:
# MY TEST

inp=[]
inp.append("four")
inp.append("five")
res = ("nine")
base=30
csp = get_problem(inp,res,base)
sol = backtracking_search(csp)
print(sol)

## Test


In [ ]:
import helper.test_q2 as q2
import time

TIME_LIMIT = 3

tests = q2.get_all_tests(prefix='q2_')
tests_passed = 0
for test in tests:
    base, operands, result = q2.scan_test_input(test)
    csp = get_problem(operands, result, base)
    start_time = time.time()
    sol = backtracking_search(csp)
    if sol is None:
        print(f'test {test} failed. time elapsed={time.time() - start_time:.5f}s')
        continue
    total_time = time.time() - start_time
    if q2.is_result_valid(sol, operands, result, base) and total_time < TIME_LIMIT:
        tests_passed += 1
        print(f'test {test} passed. time elapsed={total_time:.5f}s')
        print('solution: ')
        print('\n'.join([str(x).upper() + " " + str(y)
              for x, y in sorted(sol.items())]))
    else:
        print(f'test {test} failed. time elapsed={total_time:.5f}s')
print(f'Score = {tests_passed / len(tests) * 100}%')
